# EDA записанного датасета

Разведочный анализ локальных записей из `data/raw/Recorded`.

В папке есть два типа файлов:

```text
data/raw/Recorded/
├── Автобус 1.wav ... Автобус 8.wav
├── Клаксон 1.wav ... Клаксон 8.wav
├── Скорая 1.wav ... Скорая 8.wav
├── BUS.wav
├── Car_horn.wav
└── Siren_Ambulance.wav
```

Файлы с номерами — отдельные моно-дорожки с микрофонов. Файлы `BUS.wav`, `Car_horn.wav` и `Siren_Ambulance.wav` — собранные 8-канальные версии тех же событий.

## Настройка

In [ ]:
from pathlib import Path
import re

import IPython.display as ipd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_rows', 80)
pd.set_option('display.max_columns', 50)

RANDOM_STATE = 42
RAW_LABEL_MAP = {
    'Автобус': 'BUS',
    'Клаксон': 'Car_horn',
    'Скорая': 'Siren_Ambulance',
}
EVENT_ORDER = ['BUS', 'Car_horn', 'Siren_Ambulance']

In [ ]:
repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent

recorded_root_candidates = [
    repo_root / 'data' / 'raw' / 'Recorded',
    repo_root / 'data' / 'raw' / 'recorded',
]
RECORDED_ROOT = next((path for path in recorded_root_candidates if path.exists()), None)
if RECORDED_ROOT is None:
    raise FileNotFoundError('Не найдена папка data/raw/Recorded.')

print(f'Корень репозитория: {repo_root}')
print(f'Папка с записями: {RECORDED_ROOT}')
print(f'WAV-файлов: {len(list(RECORDED_ROOT.glob("*.wav")))}')

## Manifest

In [ ]:
def parse_recorded_filename(path: Path) -> dict:
    stem = path.stem
    mono_match = re.match(r'^(Автобус|Клаксон|Скорая)\s+(\d+)$', stem)
    if mono_match:
        raw_label, mic = mono_match.groups()
        return {
            'event': RAW_LABEL_MAP[raw_label],
            'raw_label': raw_label,
            'kind': 'mono_microphone',
            'mic': int(mic),
        }

    if stem in RAW_LABEL_MAP.values():
        return {
            'event': stem,
            'raw_label': stem,
            'kind': 'multichannel_mix',
            'mic': np.nan,
        }

    return {
        'event': stem,
        'raw_label': stem,
        'kind': 'unknown',
        'mic': np.nan,
    }


def audio_stats(path: Path) -> dict:
    info = sf.info(path)
    audio, sr = sf.read(path, always_2d=True)
    return {
        'sample_rate': info.samplerate,
        'channels': info.channels,
        'frames': info.frames,
        'duration_sec': info.duration,
        'format': info.format,
        'subtype': info.subtype,
        'rms': float(np.sqrt(np.mean(audio ** 2))),
        'peak_abs': float(np.max(np.abs(audio))),
        'mean_abs': float(np.mean(np.abs(audio))),
    }

rows = []
for path in sorted(RECORDED_ROOT.glob('*.wav')):
    rows.append(
        {
            'filename': path.name,
            'filepath': path,
            **parse_recorded_filename(path),
            **audio_stats(path),
        }
    )

manifest = pd.DataFrame(rows)
manifest['duration_min'] = manifest['duration_sec'] / 60
manifest['duration_hour'] = manifest['duration_sec'] / 3600
manifest = manifest.sort_values(['kind', 'event', 'mic', 'filename']).reset_index(drop=True)

display(manifest.drop(columns='filepath'))

В датасете три целевых события: `BUS`, `Car_horn` и `Siren_Ambulance`. Для каждого события есть 8 моно-дорожек, соответствующих 8 микрофонам, и один 8-канальный собранный файл.

## Баланс и длительность

In [ ]:
mono = manifest.query("kind == 'mono_microphone'").copy()
multichannel = manifest.query("kind == 'multichannel_mix'").copy()

mono_summary = (
    mono.groupby('event')
    .agg(
        files=('filename', 'count'),
        microphones=('mic', 'nunique'),
        duration_sec_per_mic=('duration_sec', 'first'),
        total_track_sec=('duration_sec', 'sum'),
        total_track_min=('duration_min', 'sum'),
        sample_rate_values=('sample_rate', lambda x: sorted(x.unique())),
        channel_values=('channels', lambda x: sorted(x.unique())),
    )
    .reindex(EVENT_ORDER)
    .reset_index()
)

unique_event_duration_sec = mono_summary['duration_sec_per_mic'].sum()
total_track_sec = mono_summary['total_track_sec'].sum()

print(f'Моно-файлов: {len(mono)}')
print(f'Агрегированных 8-канальных файлов: {len(multichannel)}')
print(f'Суммарная длительность дорожек: {total_track_sec:.2f} сек = {total_track_sec / 3600:.4f} часа')
print(f'Уникальная длительность событий без умножения на микрофоны: {unique_event_duration_sec:.2f} сек = {unique_event_duration_sec / 3600:.4f} часа')

display(mono_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

sns.barplot(data=mono_summary, x='event', y='files', order=EVENT_ORDER, color='#4C78A8', ax=axes[0])
axes[0].set_title('Количество моно-дорожек по событиям')
axes[0].set_xlabel('Событие')
axes[0].set_ylabel('Файлы')
axes[0].bar_label(axes[0].containers[0])

sns.barplot(data=mono_summary, x='event', y='duration_sec_per_mic', order=EVENT_ORDER, color='#F58518', ax=axes[1])
axes[1].set_title('Длительность события на одном микрофоне')
axes[1].set_xlabel('Событие')
axes[1].set_ylabel('Секунды')
axes[1].bar_label(axes[1].containers[0], fmt='%.1f')

plt.tight_layout()
plt.show()

Датасет сбалансирован по числу микрофонных дорожек: у каждого события по 8 файлов. По длительности события отличаются: `BUS` короче остальных, `Siren_Ambulance` самый длинный. Если считать каждую микрофонную дорожку как отдельный аудиофайл, доступно около 0.089 часа аудио; если считать только уникальное время событий без умножения на 8 микрофонов — около 0.011 часа.

## Технические параметры

In [ ]:
technical_summary = (
    manifest.groupby(['kind', 'event'])
    .agg(
        files=('filename', 'count'),
        channels=('channels', lambda x: sorted(x.unique())),
        sample_rates=('sample_rate', lambda x: sorted(x.unique())),
        frames_min=('frames', 'min'),
        frames_max=('frames', 'max'),
        duration_sec_min=('duration_sec', 'min'),
        duration_sec_max=('duration_sec', 'max'),
        subtypes=('subtype', lambda x: sorted(x.unique())),
    )
    .reset_index()
    .sort_values(['kind', 'event'])
)

display(technical_summary)

Все файлы имеют частоту дискретизации 44.1 кГц и PCM 16-bit. Моно-дорожки внутри одного события имеют одинаковое число frames, поэтому их можно складывать в многоканальный WAV без обрезки или padding.

## Громкость по микрофонам

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))

rms_pivot = mono.pivot(index='event', columns='mic', values='rms').reindex(EVENT_ORDER)
sns.heatmap(rms_pivot, annot=True, fmt='.3f', cmap='crest', cbar_kws={'label': 'RMS'}, ax=axes[0])
axes[0].set_title('RMS по микрофонам')
axes[0].set_xlabel('Микрофон')
axes[0].set_ylabel('Событие')

peak_pivot = mono.pivot(index='event', columns='mic', values='peak_abs').reindex(EVENT_ORDER)
sns.heatmap(peak_pivot, annot=True, fmt='.3f', cmap='rocket_r', cbar_kws={'label': 'Peak abs'}, ax=axes[1])
axes[1].set_title('Пиковая амплитуда по микрофонам')
axes[1].set_xlabel('Микрофон')
axes[1].set_ylabel('Событие')

plt.tight_layout()
plt.show()

In [ ]:
level_summary = (
    mono.groupby('event')
    .agg(
        rms_mean=('rms', 'mean'),
        rms_min=('rms', 'min'),
        rms_max=('rms', 'max'),
        peak_max=('peak_abs', 'max'),
        loudest_mic=('rms', lambda s: int(mono.loc[s.idxmax(), 'mic'])),
        quietest_mic=('rms', lambda s: int(mono.loc[s.idxmin(), 'mic'])),
    )
    .reindex(EVENT_ORDER)
    .reset_index()
)

display(level_summary)

У `BUS` заметный разброс уровня между микрофонами: 8-й микрофон самый громкий, а peak достигает 1.0, то есть есть риск клиппинга или очень близкого к нему уровня. Для `Car_horn` уровни между микрофонами более ровные. У `Siren_Ambulance` разброс умеренный, самые сильные уровни видны на 7–8 микрофонах.

## Проверка 8-канальных файлов

In [ ]:
aggregate_checks = []
for event in EVENT_ORDER:
    event_mono = mono.query('event == @event').sort_values('mic')
    event_mix = multichannel.query('event == @event')
    if event_mix.empty:
        aggregate_checks.append({'event': event, 'status': 'нет агрегированного файла'})
        continue

    mix_row = event_mix.iloc[0]
    aggregate_checks.append(
        {
            'event': event,
            'aggregate_file': mix_row['filename'],
            'mono_files': len(event_mono),
            'aggregate_channels': int(mix_row['channels']),
            'same_channel_count': int(mix_row['channels']) == len(event_mono),
            'mono_frames_min': int(event_mono['frames'].min()),
            'mono_frames_max': int(event_mono['frames'].max()),
            'aggregate_frames': int(mix_row['frames']),
            'same_frames': int(mix_row['frames']) == int(event_mono['frames'].min()) == int(event_mono['frames'].max()),
            'duration_sec': mix_row['duration_sec'],
        }
    )

aggregate_checks = pd.DataFrame(aggregate_checks)
display(aggregate_checks)

Агрегированные файлы корректно соответствуют исходным дорожкам: в каждом 8 каналов, число channels равно числу микрофонов, а длительность совпадает с моно-дорожками соответствующего события.

## Волновые формы

In [ ]:
fig, axes = plt.subplots(len(EVENT_ORDER), 1, figsize=(14, 8), sharex=False)

for ax, event in zip(axes, EVENT_ORDER):
    row = mono.query('event == @event and mic == 1').iloc[0]
    signal, sr = librosa.load(row['filepath'], sr=None, mono=True)
    librosa.display.waveshow(signal, sr=sr, ax=ax)
    ax.set_title(f'{event}: микрофон 1')
    ax.set_xlabel('Время, сек')
    ax.set_ylabel('Амплитуда')

plt.tight_layout()
plt.show()

## Mel-спектрограммы

In [ ]:
fig, axes = plt.subplots(len(EVENT_ORDER), 1, figsize=(14, 9), sharex=False)

for ax, event in zip(axes, EVENT_ORDER):
    row = mono.query('event == @event and mic == 1').iloc[0]
    signal, sr = librosa.load(row['filepath'], sr=None, mono=True)
    mel = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=96)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel', cmap='magma', ax=ax)
    ax.set_title(f'{event}: mel-спектрограмма, микрофон 1')
    ax.set_xlabel('Время, сек')
    ax.set_ylabel('Mel')

plt.tight_layout()
plt.show()

Визуально события различаются по временной структуре: `BUS` короткий и плотный, `Car_horn` содержит более выраженные устойчивые тональные участки, а `Siren_Ambulance` дольше и имеет периодичную структуру сирены. Для классификации здесь полезны спектральные признаки, но объём данных очень мал, поэтому такой набор лучше использовать как небольшой локальный sanity-check, а не как самостоятельный обучающий датасет.

## Прослушивание примера

In [ ]:
sample = mono.query('event == "Siren_Ambulance" and mic == 1').iloc[0]
print(sample[['filename', 'event', 'mic', 'duration_sec', 'sample_rate']])
ipd.Audio(sample['filepath'])

## Итоги

- В папке `Recorded` лежит 24 моно-дорожки: 3 события × 8 микрофонов.
- Дополнительно есть 3 собранных 8-канальных WAV, по одному на событие.
- Все записи имеют 44.1 кГц, PCM 16-bit; технически они совместимы для многоканальной обработки.
- Уникальная длительность событий без учёта повторения по микрофонам — около 40.2 секунды. Суммарная длительность всех моно-дорожек — около 321.6 секунды.
- Главный риск данных — очень маленький объём и неодинаковая громкость между микрофонами, особенно у `BUS`. Перед обучением стоит нормализовать уровни или явно учитывать микрофонный канал.